# GLM Algorithm Arena — Phase 2: Multi-Dataset Benchmarking

Benchmarking of **all GLM solvers** across **real and synthetic datasets**
with a comprehensive, dataset-adaptive hyperparameter grid.

Each *(solver, config)* pair produces one coefficient vector β̂.
t-SNE and UMAP projections reveal how algorithmically similar the solutions
are across solver families.

Key features of this notebook:
* **≥ 20 hyperparameter variations per solver** (lambda values scaled to each
  dataset's natural scale; secondary parameters swept over representative grids).
* Configurations **capped at 50 per solver** to keep runtimes bounded.
* Embedding plots coloured by **individual solver** (20 distinct colours, tab20).


## Section 0 — Setup

In [ ]:
import sys, os, time, warnings, itertools
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import (
    load_diabetes, load_breast_cancer, load_iris, load_wine,
    make_regression, make_classification, make_friedman1,
)
from sklearn.manifold import TSNE
import sklearn
import umap

print(f"sklearn  {sklearn.__version__}")
print(f"umap     {umap.__version__}")

_sk_ver = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
_TSNE_ITER_KW = 'max_iter' if _sk_ver >= (1, 5) else 'n_iter'

# ── Solver imports ────────────────────────────────────────────────────────────
from src.glmzoo.solvers.classical.ols             import OLSSolver
from src.glmzoo.solvers.classical.ridge           import RidgeSolver
from src.glmzoo.solvers.classical.glm_irls        import GLMIRLSSolver
from src.glmzoo.solvers.penalized.lasso_cd        import LassoCDSolver
from src.glmzoo.solvers.penalized.elastic_net     import ElasticNetSolver
from src.glmzoo.solvers.penalized.adaptive_lasso  import AdaptiveLassoSolver
from src.glmzoo.solvers.penalized.scad_lla        import SCADLLASolver
from src.glmzoo.solvers.penalized.mcp_cd          import MCPCDSolver
from src.glmzoo.solvers.penalized.group_lasso     import GroupLassoSolver
from src.glmzoo.solvers.penalized.fused_lasso     import FusedLassoSolver
from src.glmzoo.solvers.path.lars                 import LARSSolver
from src.glmzoo.solvers.first_order.ista          import ISTASolver
from src.glmzoo.solvers.first_order.fista         import FISTASolver
from src.glmzoo.solvers.online.sgd                import SGDSolver
from src.glmzoo.solvers.online.implicit_sgd       import ImplicitSGDSolver
from src.glmzoo.solvers.online.adagrad            import AdaGradSolver
from src.glmzoo.solvers.online.fobos              import FOBOSSolver
from src.glmzoo.solvers.online.rda                import RDASolver
from src.glmzoo.solvers.online.truncated_gradient import TruncatedGradientSolver
from src.glmzoo.solvers.online.renewable_glm      import RenewableGLMSolver

print("All solver classes imported.")

# ── Family mapping ────────────────────────────────────────────────────────────
FAMILY = {
    OLSSolver: "Classical",           GLMIRLSSolver: "Classical",
    LARSSolver: "Classical",          RidgeSolver: "Penalized",
    LassoCDSolver: "Penalized",       ElasticNetSolver: "Penalized",
    AdaptiveLassoSolver: "Penalized", SCADLLASolver: "Penalized",
    MCPCDSolver: "Penalized",         GroupLassoSolver: "Penalized",
    FusedLassoSolver: "Penalized",    ISTASolver: "First-order",
    FISTASolver: "First-order",       SGDSolver: "Online",
    ImplicitSGDSolver: "Online",      AdaGradSolver: "Online",
    FOBOSSolver: "Online",            RDASolver: "Online",
    TruncatedGradientSolver: "Online", RenewableGLMSolver: "Online",
}

FAMILY_COLORS = {
    "Classical":   "#1f77b4",
    "Penalized":   "#2ca02c",
    "First-order": "#ff7f0e",
    "Online":      "#d62728",
}

# ── Per-solver colours (tab20, 20 distinct) ───────────────────────────────────
_SOLVER_ORDER = [
    OLSSolver, GLMIRLSSolver, LARSSolver,
    RidgeSolver, LassoCDSolver, ElasticNetSolver,
    AdaptiveLassoSolver, SCADLLASolver, MCPCDSolver,
    GroupLassoSolver, FusedLassoSolver,
    ISTASolver, FISTASolver,
    SGDSolver, ImplicitSGDSolver, AdaGradSolver,
    FOBOSSolver, RDASolver, TruncatedGradientSolver, RenewableGLMSolver,
]
ALGO_COLORS = {cls.__name__: plt.cm.tab20(i / 20) for i, cls in enumerate(_SOLVER_ORDER)}
ALGO_LABELS = {
    "OLSSolver":               "OLS",
    "GLMIRLSSolver":           "GLM-IRLS",
    "LARSSolver":              "LARS",
    "RidgeSolver":             "Ridge",
    "LassoCDSolver":           "Lasso-CD",
    "ElasticNetSolver":        "Elastic Net",
    "AdaptiveLassoSolver":     "Adaptive Lasso",
    "SCADLLASolver":           "SCAD",
    "MCPCDSolver":             "MCP-CD",
    "GroupLassoSolver":        "Group Lasso",
    "FusedLassoSolver":        "Fused Lasso",
    "ISTASolver":              "ISTA",
    "FISTASolver":             "FISTA",
    "SGDSolver":               "SGD",
    "ImplicitSGDSolver":       "Implicit SGD",
    "AdaGradSolver":           "AdaGrad",
    "FOBOSSolver":             "FOBOS",
    "RDASolver":               "RDA",
    "TruncatedGradientSolver": "Trunc. Grad.",
    "RenewableGLMSolver":      "Renewable GLM",
}


## Section 1 — Load Datasets

In [ ]:
RNG   = np.random.default_rng(42)
MAX_N = 2000
datasets = {}

def _add_regression(X, y, name):
    X = np.array(X, dtype=float); y = np.array(y, dtype=float)
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X, y = X[ok], y[ok]
    if len(X) < 50:
        print(f"  SKIP {name}: n={len(X)} < 50"); return
    if len(X) > MAX_N:
        idx = RNG.choice(len(X), MAX_N, replace=False); X, y = X[idx], y[idx]
    X = StandardScaler().fit_transform(X)
    y = (y - y.mean()) / (y.std() + 1e-12)
    datasets[name] = dict(X=X, y=y, link='identity', kind='regression')
    print(f"  [R] {name:35s}  n={X.shape[0]:4d}  p={X.shape[1]}")

def _add_logistic(X, y, name):
    X = np.array(X, dtype=float); y = np.array(y, dtype=float)
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X, y = X[ok], y[ok]
    if len(X) < 50:
        print(f"  SKIP {name}: n={len(X)} < 50"); return
    if len(X) > MAX_N:
        idx = RNG.choice(len(X), MAX_N, replace=False); X, y = X[idx], y[idx]
    X = StandardScaler().fit_transform(X)
    datasets[name] = dict(X=X, y=y, link='logit', kind='logistic')
    print(f"  [L] {name:35s}  n={X.shape[0]:4d}  p={X.shape[1]}")

# ── A. sklearn bundled (always available offline) ─────────────────────────────
print("sklearn built-ins:")
d = load_diabetes();      _add_regression(d.data, d.target, "Diabetes")
d = load_breast_cancer(); _add_logistic(d.data, d.target.astype(float), "Breast Cancer")
d = load_iris();          _add_logistic(d.data, (d.target != 0).astype(float), "Iris (binary)")
d = load_wine();          _add_logistic(d.data, (d.target == 0).astype(float), "Wine (binary)")

# ── B. statsmodels bundled ────────────────────────────────────────────────────
print("\nstatsmodels built-ins:")
try:
    import statsmodels.datasets as smd

    d = smd.fair.load_pandas().data
    _add_regression(d.drop(columns=['affairs']).select_dtypes(include=[float,int]).values,
                    d['affairs'].values, "Fair-Affairs")

    d = smd.macrodata.load_pandas().data
    _add_regression(d.drop(columns=['realgdp']).select_dtypes(include=[float,int]).values,
                    d['realgdp'].values, "Macro GDP")

    d = smd.star98.load_pandas().data.select_dtypes(include=[float,int]).dropna()
    y0 = d.iloc[:,0].values; X0 = d.iloc[:,1:].values
    _add_logistic(X0, (y0 > np.median(y0)).astype(float), "STAR98 (pass)")

    d = smd.anes96.load_pandas().data.select_dtypes(include=[float,int]).dropna()
    y0 = d.iloc[:,0].values; X0 = d.iloc[:,1:].values
    _add_logistic(X0, (y0 > np.median(y0)).astype(float), "ANES96 (vote)")

    d = smd.grunfeld.load_pandas().data.select_dtypes(include=[float,int]).dropna()
    _add_regression(d.drop(columns=['invest']).values, d['invest'].values, "Grunfeld")

    d = smd.modechoice.load_pandas().data.select_dtypes(include=[float,int]).dropna()
    y0 = d.iloc[:,0].values; X0 = d.iloc[:,1:].values
    _add_logistic(X0, (y0 > np.median(y0)).astype(float), "Mode Choice")

    d = smd.randhie.load_pandas().data.select_dtypes(include=[float,int]).dropna()
    yc = 'lnmeddol' if 'lnmeddol' in d.columns else d.columns[0]
    _add_regression(d.drop(columns=[yc]).values, d[yc].values, "RAND-HIE")
except Exception as e:
    print(f"  statsmodels error: {e}")

# ── C. Synthetic datasets (fill to ~20 total) ─────────────────────────────────
print("\nSynthetic:")
if len(datasets) < 20:
    Xs, ys = make_friedman1(n_samples=1000, n_features=10, noise=0.5, random_state=42)
    _add_regression(Xs, ys, "Synth-Friedman1")
if len(datasets) < 20:
    Xs, ys = make_regression(n_samples=800, n_features=20, n_informative=10, noise=0.5, random_state=1)
    _add_regression(Xs, ys, "Synth-Reg-Dense")
if len(datasets) < 20:
    Xs, ys = make_regression(n_samples=500, n_features=100, n_informative=12, noise=1.0, random_state=2)
    _add_regression(Xs, ys, "Synth-Reg-Sparse")
if len(datasets) < 20:
    Xs, ys = make_regression(n_samples=600, n_features=30, n_informative=20, effective_rank=10, noise=0.4, random_state=3)
    _add_regression(Xs, ys, "Synth-Reg-Corr")
if len(datasets) < 20:
    Xs, ys = make_regression(n_samples=300, n_features=150, n_informative=15, noise=1.5, random_state=4)
    _add_regression(Xs, ys, "Synth-Reg-HighDim")
if len(datasets) < 20:
    Xs, ys = make_classification(n_samples=1000, n_features=25, n_informative=15, n_redundant=5, random_state=5)
    _add_logistic(Xs, ys.astype(float), "Synth-Logit-Dense")
if len(datasets) < 20:
    Xs, ys = make_classification(n_samples=500, n_features=80, n_informative=10, n_redundant=5, random_state=6)
    _add_logistic(Xs, ys.astype(float), "Synth-Logit-Sparse")
if len(datasets) < 20:
    Xs, ys = make_classification(n_samples=800, n_features=40, n_informative=20, n_redundant=10, flip_y=0.1, random_state=7)
    _add_logistic(Xs, ys.astype(float), "Synth-Logit-Noisy")

print(f"\nTotal datasets loaded: {len(datasets)}")


## Section 2 — Adaptive Hyperparameter Grid

For every dataset we compute `lam_max = max|X'(y-ȳ)|/n` and generate lambda
grids as fractions of that value.  This ensures the regularisation path covers
the full range from near-OLS to highly-sparse, adapted to each dataset's scale.

| Solver | Configs | Strategy |
|---|---|---|
| Ridge, Lasso-CD, ISTA, FISTA, LARS | 25 | 25 log-spaced λ ∈ [0.005·λ_max, λ_max] |
| Elastic Net | 25 | 5 λ × 5 α (l1-ratio) |
| Adaptive Lasso, SCAD, MCP | 20 | 5 λ × 4 secondary param |
| Group Lasso | 20 | 5 λ × 4 group configs |
| Fused Lasso | 25 | 5×5 (λ1, λ2) grid |
| SGD, Implicit SGD, AdaGrad | 20 | 5 step-size × 4 n_passes |
| FOBOS, RDA, Trunc. Grad. | 24 | 4 λ × 3 γ₀ × 2 K/passes |
| Renewable GLM | 12 | 4 n_batches × 3 ridge_init |


In [ ]:
def compute_lam_max(X, y):
    return float(np.max(np.abs(X.T @ (y - y.mean()))) / len(y))

def make_groups(p, n_groups):
    n_groups = max(1, min(n_groups, p))
    return [list(range(j, p, n_groups)) for j in range(n_groups)]

def build_configs(lam_max, p, kind='regression'):
    '''≥ 20 (solver, config, label) triples per solver, capped at 50.'''
    cfgs = []

    def lam_grid(n=25, frac_min=0.005):
        lo = max(lam_max * frac_min, 1e-7)
        hi = max(lam_max, lo * 2)
        return np.logspace(np.log10(lo), np.log10(hi), n)

    cfgs.append((OLSSolver,     {}, 'OLS'))
    cfgs.append((GLMIRLSSolver, {}, 'IRLS'))

    if kind == 'regression':
        for lam in lam_grid(25): cfgs.append((LARSSolver, {'lam': float(lam)}, f'LARS λ={lam:.4g}'))

    for lam in lam_grid(25): cfgs.append((RidgeSolver,   {'lam': float(lam)}, f'Ridge λ={lam:.4g}'))
    for lam in lam_grid(25): cfgs.append((LassoCDSolver, {'lam': float(lam)}, f'Lasso λ={lam:.4g}'))
    for lam in lam_grid(25): cfgs.append((ISTASolver,    {'lam': float(lam)}, f'ISTA λ={lam:.4g}'))
    for lam in lam_grid(25): cfgs.append((FISTASolver,   {'lam': float(lam)}, f'FISTA λ={lam:.4g}'))

    for lam in lam_grid(5):
        for a in [0.1, 0.3, 0.5, 0.7, 0.9]:
            cfgs.append((ElasticNetSolver, {'lam': float(lam), 'alpha': a}, f'EN λ={lam:.4g} α={a}'))

    for lam in lam_grid(5):
        for g in [0.5, 1.0, 2.0, 3.0]:
            cfgs.append((AdaptiveLassoSolver, {'lam': float(lam), 'gamma': g}, f'AdaLasso λ={lam:.4g} γ={g}'))

    for lam in lam_grid(5):
        for a in [2.5, 3.7, 5.0, 7.0]:
            cfgs.append((SCADLLASolver, {'lam': float(lam), 'a': a}, f'SCAD λ={lam:.4g} a={a}'))

    for lam in lam_grid(5):
        for g in [1.5, 2.0, 3.0, 5.0]:
            cfgs.append((MCPCDSolver, {'lam': float(lam), 'gamma': g}, f'MCP λ={lam:.4g} γ={g}'))

    ng_vals = sorted(set([2, max(2, p//5), max(2, p//3), min(p, max(5, p//2))]))[:4]
    for lam in lam_grid(5):
        for ng in ng_vals:
            cfgs.append((GroupLassoSolver, {'lam': float(lam), 'groups': make_groups(p, ng)},
                         f'GrpLasso λ={lam:.4g} G={ng}'))

    if kind == 'regression':
        lf = lam_grid(5)
        for l1, l2 in itertools.product(lf, lf):
            cfgs.append((FusedLassoSolver, {'lam1': float(l1), 'lam2': float(l2)},
                         f'Fused λ1={l1:.4g} λ2={l2:.4g}'))

    for g in [0.001, 0.005, 0.01, 0.05, 0.1]:
        for T in [5, 10, 20, 50]:
            cfgs.append((SGDSolver, {'gamma0': g, 'n_passes': T}, f'SGD γ={g} T={T}'))

    for g in [0.001, 0.005, 0.01, 0.05, 0.1]:
        for a in [0.51, 0.6, 0.75, 0.9]:
            cfgs.append((ImplicitSGDSolver, {'gamma0': g, 'alpha': a, 'n_passes': 10}, f'ISGD γ={g} α={a}'))

    for e in [0.01, 0.05, 0.1, 0.5, 1.0]:
        for T in [5, 10, 20, 50]:
            cfgs.append((AdaGradSolver, {'eta': e, 'n_passes': T}, f'AdaGrad η={e} T={T}'))

    for lam in lam_grid(4):
        for g in [0.01, 0.05, 0.1]:
            for T in [10, 20]:
                cfgs.append((FOBOSSolver, {'lam': float(lam), 'gamma0': g, 'n_passes': T},
                             f'FOBOS λ={lam:.4g} γ={g} T={T}'))

    for lam in lam_grid(4):
        for g in [0.01, 0.05, 0.1]:
            for T in [5, 10]:
                cfgs.append((RDASolver, {'lam': float(lam), 'gamma0': g, 'n_passes': T},
                             f'RDA λ={lam:.4g} γ={g} T={T}'))

    for lam in lam_grid(4):
        for g in [0.01, 0.05, 0.1]:
            for K in [5, 10]:
                cfgs.append((TruncatedGradientSolver, {'lam': float(lam), 'gamma0': g, 'K': K},
                             f'TruncGrad λ={lam:.4g} γ={g} K={K}'))

    for nb in [5, 10, 20, 50]:
        for ri in [1e-4, 1e-3, 1e-2]:
            cfgs.append((RenewableGLMSolver, {'n_batches': nb, 'ridge_init': ri},
                         f'Renew B={nb} r={ri}'))

    return cfgs

print(f"Config builder defined.")
print(f"Example: Diabetes gets "
      f"{len(build_configs(0.5, 10, 'regression'))} configs (regression, p=10)")


## Section 3 — Run Arena

In [ ]:
SKIP_FOR_LOGISTIC = {LARSSolver, FusedLassoSolver}
records = []
t_arena = time.time()

for di, (ds_name, ds) in enumerate(datasets.items()):
    X, y, link, kind = ds['X'], ds['y'], ds['link'], ds['kind']
    n, p = X.shape
    lam_max      = compute_lam_max(X, y)
    solver_cfgs  = build_configs(lam_max, p, kind)
    t0 = time.time(); n_ok = 0
    for SolverCls, cfg, label in solver_cfgs:
        if kind == 'logistic' and SolverCls in SKIP_FOR_LOGISTIC:
            continue
        try:
            res  = SolverCls(config=cfg).fit(X, y, link=link)
            beta = res.beta_hat
            if not np.all(np.isfinite(beta)):
                raise ValueError('non-finite beta')
            records.append(dict(
                dataset=ds_name, kind=kind, label=label,
                cls=SolverCls.__name__, family=FAMILY[SolverCls],
                beta=beta.copy(), bnorm=float(np.linalg.norm(beta)),
                ok=True, p=p,
            ))
            n_ok += 1
        except Exception:
            records.append(dict(
                dataset=ds_name, kind=kind, label=label,
                cls=SolverCls.__name__, family=FAMILY.get(SolverCls, '?'),
                beta=None, bnorm=np.nan, ok=False, p=p,
            ))
    print(f"[{di+1:2d}/{len(datasets)}] {ds_name:35s}  "
          f"n={n:4d} p={p:3d}  cfgs={len(solver_cfgs):3d}  ok={n_ok:3d}  {time.time()-t0:.1f}s")

df_all = pd.DataFrame(records)
print(f"\nArena done in {time.time()-t_arena:.1f}s")
print(f"Total: {len(df_all)}  |  OK: {df_all.ok.sum()} ({100*df_all.ok.mean():.1f}%)")


## Section 4 — Results Summary

In [ ]:
print(df_all.groupby(['dataset','ok']).size().unstack(fill_value=0).to_string())
print()
print(df_all.groupby(['family','ok']).size().unstack(fill_value=0).to_string())


## Section 5 — Outlier Filtering (5× median norm)

In [ ]:
def filter_outliers(df, thresh=5.0):
    parts = []
    for ds_name in df['dataset'].unique():
        g = df[(df['dataset'] == ds_name) & df['ok']].copy()
        if len(g) == 0: continue
        med = g['bnorm'].median(); med = med if med > 0 else 1.0
        mask  = g['bnorm'] <= thresh * med
        n_out = (~mask).sum()
        if n_out: print(f"  {ds_name}: {n_out} outliers removed")
        parts.append(g[mask])
    return pd.concat(parts, ignore_index=True)

df_clean = filter_outliers(df_all)
print(f"\nClean runs: {len(df_clean)}  across {df_clean['dataset'].nunique()} datasets")


## Section 6 — Per-Dataset Embeddings: t-SNE and UMAP

Each point is one *(solver, config)* run.  **Colour = individual solver** (tab20,
20 distinct colours).  Centroid labels mark the cluster centre of each solver.


In [ ]:
embed_figs = []

for ds_name, grp in df_clean.groupby('dataset'):
    n_pts = len(grp)
    if n_pts < 10:
        print(f"SKIP {ds_name}: only {n_pts} clean points"); continue

    p       = int(grp['p'].iloc[0])
    betas   = np.vstack(grp['beta'].values)
    cls_arr = grp['cls'].values

    # t-SNE
    perp    = min(30, max(5, n_pts // 5))
    xy_tsne = TSNE(n_components=2, perplexity=perp, init='pca',
                   random_state=42, **{_TSNE_ITER_KW: 1200}).fit_transform(betas)
    # UMAP
    nn      = min(20, max(3, n_pts // 6))
    xy_umap = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.08,
                        random_state=42).fit_transform(betas)

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.patch.set_facecolor('#f5f5f5')
    ds_meta = datasets[ds_name]
    fig.suptitle(
        f"{ds_name}  ·  n={ds_meta['X'].shape[0]}, p={p}, {ds_meta['kind']}"
        f"  ·  {n_pts} (solver × config) points",
        fontsize=14, fontweight='bold', y=1.005,
    )

    present_cls = [c for c in [s.__name__ for s in _SOLVER_ORDER] if c in cls_arr]

    for ax, coords, method in zip(axes, [xy_tsne, xy_umap], ['t-SNE', 'UMAP']):
        ax.set_facecolor('#fafafa')
        for cls_name in present_cls:
            m = cls_arr == cls_name
            if m.sum() == 0: continue
            ax.scatter(coords[m,0], coords[m,1],
                       c=[ALGO_COLORS[cls_name]],
                       label=ALGO_LABELS.get(cls_name, cls_name),
                       alpha=0.72, s=45, linewidths=0, zorder=3)
        for cls_name in present_cls:
            m = cls_arr == cls_name
            if m.sum() < 3: continue
            cx, cy = coords[m,0].mean(), coords[m,1].mean()
            ax.annotate(ALGO_LABELS.get(cls_name, cls_name), (cx, cy),
                        ha='center', va='center', fontsize=6, fontweight='bold',
                        color=ALGO_COLORS[cls_name],
                        bbox=dict(boxstyle='round,pad=0.18', fc='white', alpha=0.82, ec='none'),
                        zorder=5)
        ax.set_title(method, fontsize=13, pad=10, fontweight='semibold')
        ax.set_xlabel(f'{method} 1', fontsize=9, color='#555')
        ax.set_ylabel(f'{method} 2', fontsize=9, color='#555')
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        for sp in ax.spines.values(): sp.set_visible(False)

    handles = [mpatches.Patch(color=ALGO_COLORS[c], label=ALGO_LABELS.get(c, c))
               for c in present_cls]
    axes[1].legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1.0),
                   framealpha=0.92, fontsize=8.5, title='Solver', title_fontsize=9,
                   borderpad=0.7, labelspacing=0.35, edgecolor='#ccc')

    plt.tight_layout()
    safe  = (ds_name.replace(' ','_').replace('/','_')
             .replace('(','').replace(')','').replace('-','_'))
    fname = f'fig_embed_{safe}.png'
    plt.savefig(fname, dpi=110, bbox_inches='tight')
    plt.show()
    embed_figs.append(fname)
    print(f'  Saved: {fname}')

print(f"\nTotal embedding figures: {len(embed_figs)}")


## Section 7 — Coefficient Norm by Solver Family

In [ ]:
fam_order = ['Classical', 'Penalized', 'First-order', 'Online']
data_bp   = [df_clean[df_clean['family'] == f]['bnorm'].dropna().values for f in fam_order]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#f5f5f5'); ax.set_facecolor('#fafafa')
bp = ax.boxplot(data_bp, labels=fam_order, patch_artist=True, notch=False,
                medianprops=dict(color='k', lw=2.2),
                whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2),
                flierprops=dict(marker='o', markersize=3, alpha=0.35))
for patch, fam in zip(bp['boxes'], fam_order):
    patch.set_facecolor(FAMILY_COLORS[fam]); patch.set_alpha(0.72)
ax.set_ylabel('‖β̂‖₂', fontsize=12)
ax.set_title('Coefficient Norm Distribution by Solver Family (all datasets)',
             fontsize=12, fontweight='semibold')
ax.grid(axis='y', alpha=0.35, zorder=0)
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_norm_boxplot.png', dpi=110, bbox_inches='tight')
plt.show()
print("Saved: fig_norm_boxplot.png")


## Section 8 — Summary

In [ ]:
print(f"Datasets loaded:      {len(datasets)}")
print(f"Total arena runs:     {len(df_all)}")
print(f"After outlier filter: {len(df_clean)}")
print()
print("Success rate by family:")
for fam in ['Classical', 'Penalized', 'First-order', 'Online']:
    sub = df_all[df_all['family'] == fam]
    if len(sub) == 0: continue
    print(f"  {fam:12s}: {sub['ok'].sum():5d}/{len(sub):5d}  ({100*sub['ok'].mean():.1f}%)")
print()
print("Mean ‖β̂‖₂ after filtering:")
for fam in ['Classical', 'Penalized', 'First-order', 'Online']:
    sub = df_clean[df_clean['family'] == fam]['bnorm']
    if len(sub) == 0: continue
    print(f"  {fam:12s}: mean={sub.mean():.4f}  median={sub.median():.4f}")
print()
print("Solver config counts (median per dataset):")
for cls in _SOLVER_ORDER:
    n_cfg = df_all[df_all['cls'] == cls.__name__].groupby('dataset')['label'].nunique().median()
    print(f"  {ALGO_LABELS[cls.__name__]:20s}: ~{n_cfg:.0f} configs/dataset")
